# 08. 계층적 에이전트 팀 (Hierarchical Agent Team)

> Supervisor도 부담을 느낄 만큼 큰 작업은 **팀 단위로 쪼개요**. Research Team / Doc Writing Team을 서브그래프로 만들어 Super-Supervisor가 조율하는 구조를 구현해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **계층적 멀티 에이전트** 구조가 필요한 이유를 설명하고, 단일 Supervisor와의 차이를 구분할 수 있어요
2. **Research Team**과 **Doc Writing Team**을 독립적인 서브그래프로 구성하고 각 팀에 특화된 도구를 연결할 수 있어요
3. **AgentFactory**, **create_team_supervisor**, **preprocess** 헬퍼 패턴을 사용해 에이전트 생성 코드를 재사용할 수 있어요
4. **Super-Graph**에서 파이프라인 연산자(`|`)로 서브그래프를 합성하고, `get_last_message`/`join_graph` 헬퍼로 상태를 변환할 수 있어요

## 사전 지식

- Part 09의 02-Supervisor, 03-Multi-Agent-Collaboration, 04-Multi-Agent-Supervisor 노트북
- StateGraph, 조건부 엣지, MemorySaver 사용법
- `create_agent` 및 `create_team_supervisor` 패턴
- 이전 노트북 `07-Skills-Pattern.ipynb`에서 배운 온디맨드 컨텍스트 로딩 개념

## 계층적 에이전트 팀이란?

이전 Supervisor 패턴에서는 **하나의 Supervisor**가 모든 작업자를 직접 관리했어요. 이 방식은 단순한 경우에 효율적이지만, 다음 상황에서는 한계가 생겨요.

| 문제 상황 | 설명 |
|-----------|------|
| **작업자 수 증가** | 작업자가 10개 이상이 되면 Supervisor의 컨텍스트 부담이 커져요 |
| **전문 영역 분리** | 웹 연구 vs 문서 작성처럼 완전히 다른 도구셋이 필요한 경우 |
| **독립적 서브워크플로** | 각 팀이 자체 루프를 가질 때 단일 그래프로 표현하기 어려워요 |

> 💡 **핵심 정리**: 계층적 구조는 **대기업의 조직 체계**와 같아요. CEO(Top Supervisor)가 직원 100명을 직접 관리하면 엉망이 되죠. 대신 CEO → 부서장(Sub-Supervisor) → 팀원(Worker) 구조로 위임하면 효율적이에요. CEO는 "조사 부서에서 자료 모아오고, 집필 부서에서 보고서 써"라고만 지시하면 돼요. 각 부서 내부에서 누가 무엇을 할지는 부서장이 알아서 결정해요.

> 🔑 **핵심 개념**: 계층적 에이전트 팀은 **책임의 위임**이 핵심이에요. 최상위 감독자는 "어떤 팀이 할 것인지"만 결정하고, 팀 내부의 "어떻게 할 것인지"는 Sub-Supervisor가 결정해요.

이 노트북에서 구현할 시스템 구조를 살펴볼게요.

```mermaid
flowchart TD
    User(["사용자 요청<br/>User Request"]):::input
    
    subgraph SuperGraph["Super-Graph (최상위 그래프)"]
        TopSup["총 감독자<br/>Top Supervisor"]:::process
        
        subgraph ResearchTeam["Research Team"]
            RSup["Research Supervisor"]:::process
            Searcher["Searcher<br/>(TavilySearch)"]:::process
            Scraper["WebScraper<br/>(URL 스크래핑)"]:::process
            RSup -->|다음 작업자 선택| Searcher
            RSup -->|다음 작업자 선택| Scraper
            Searcher --> RSup
            Scraper --> RSup
        end
        
        subgraph DocTeam["Doc Writing Team"]
            DSup["Doc Supervisor"]:::process
            Writer["DocWriter<br/>(문서 작성/편집)"]:::process
            Note["NoteTaker<br/>(개요 작성)"]:::process
            Chart["ChartGenerator<br/>(차트 생성)"]:::process
            DSup -->|다음 작업자 선택| Writer
            DSup -->|다음 작업자 선택| Note
            DSup -->|다음 작업자 선택| Chart
            Writer --> DSup
            Note --> DSup
            Chart --> DSup
        end
        
        TopSup -->|ResearchTeam| ResearchTeam
        TopSup -->|PaperWritingTeam| DocTeam
        ResearchTeam --> TopSup
        DocTeam --> TopSup
        TopSup -->|FINISH| Done
    end
    
    User --> TopSup
    Done(["최종 결과<br/>Final Output"]):::output
    
    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

> 💡 **실무 팁**: 다이어그램에서 ResearchTeam 전체와 DocTeam 전체가 각각 하나의 "노드"처럼 Super-Graph에 연결되는 것을 주목해요. 팀 내부의 복잡한 루프는 Super-Graph에서 완전히 숨겨져요. 이것이 계층적 설계의 핵심 장점인 **캡슐화(Encapsulation)**예요.

## 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 API 키를 읽어와요
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택)
# ---------------------------------------------------
# 실행 흐름을 LangSmith에서 시각적으로 확인할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Hierarchical-Agent"

### 1-2. Research Team 도구

| 도구 | 설명 |
|------|------|
| `TavilySearch` | 최신 웹 검색 수행 (Tavily API) |
| `scrape_webpages` | URL 목록에서 전체 페이지 내용 스크래핑 |

> 💡 **실무 팁**: `TavilySearch`는 간결한 검색 결과를 반환하고, `scrape_webpages`는 특정 URL의 전체 내용을 가져와요. 두 도구를 조합하면 "검색 후 상세 조사" 워크플로를 구현할 수 있어요.

In [ ]:
from typing import List

from langchain_community.document_loaders import WebBaseLoader
from langchain_tavily import TavilySearch
from langchain.tools import tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 1-2. Doc Writing Team 도구

문서 작성 팀은 파일 시스템에 접근하는 도구를 사용해요.

| 도구 | 기능 |
|------|------|
| `create_outline` | 개요(아웃라인)를 파일로 저장 |
| `write_document` | 새 문서 작성 및 저장 |
| `read_document` | 저장된 문서 읽기 (부분 읽기 지원) |
| `edit_document` | 특정 줄 번호에 텍스트 삽입 |
| `PythonREPLTool` | Python 코드 실행 (차트 생성 등) |

> ⚠️ **자주 하는 실수**: `WORKING_DIRECTORY`를 설정하지 않으면 에이전트가 파일을 저장할 위치를 모를 수 있어요. 반드시 작업 디렉토리를 명시적으로 만들어 주세요.

In [ ]:
from pathlib import Path
from typing import Dict, Optional
from typing_extensions import Annotated

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_experimental.tools import PythonREPLTool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 유틸리티 함수 및 헬퍼 정의

에이전트 팀을 효율적으로 구축하기 위한 재사용 가능한 컴포넌트를 정의해요.

### 2-1. AgentFactory 클래스

동일한 LLM 설정으로 여러 에이전트를 만들 때 코드 중복을 줄여주는 팩토리 패턴이에요.

> 🔑 **핵심 개념**: `create_agent_node`는 에이전트의 반환값을 `HumanMessage`로 감싸서 반환해요. 이렇게 하면 어떤 팀 멤버가 메시지를 남겼는지 `name` 필드로 추적할 수 있어요.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 2-2. create_team_supervisor 함수

팀 감독자(Sub-Supervisor)를 생성하는 핵심 함수예요. 이 함수가 반환하는 체인은 `RouteResponse`라는 구조화된 출력으로 다음 작업자를 선택해요.

> 💡 **핵심 정리**: `Literal[*options_for_next]`를 사용하면 Pydantic이 허용된 값만 받도록 강제해요. LLM이 임의의 문자열을 반환할 수 없어서 라우팅이 훨씬 안정적이에요.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from pydantic import BaseModel
from typing import Literal

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. Research Team 구성

Research Team은 **Searcher**와 **WebScraper** 두 작업자와 **Research Supervisor**로 구성돼요.

```mermaid
flowchart LR
    START([시작]):::input
    RSup["Research Supervisor"]:::process
    Searcher["Searcher<br>(TavilySearch)"]:::process
    Scraper["WebScraper<br>(scrape_webpages)"]:::process
    END_NODE(["END"]):::output

    START --> RSup
    RSup -->|Searcher 선택| Searcher
    RSup -->|WebScraper 선택| Scraper
    RSup -->|FINISH| END_NODE
    Searcher --> RSup
    Scraper --> RSup

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

In [ ]:
from typing import List, TypedDict
from typing_extensions import Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → Supervisor → (Searcher | WebScraper | END) → Supervisor → ...
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. Doc Writing Team 구성

Doc Writing Team은 **NoteTaker(개요 작성)→DocWriter(내용 작성)→ChartGenerator(차트 생성)→DocWriter(최종 저장)** 순서로 동작해요.

### 상태 전처리(preprocess) 패턴

에이전트가 현재 작업 디렉토리에 어떤 파일이 있는지 알아야 이어서 작업할 수 있어요. `preprocess` 함수를 파이프라인 연산자(`|`)로 에이전트 앞에 연결하면 상태를 자동으로 보강할 수 있어요.

> 🔑 **핵심 개념**: `preprocess | agent` 패턴에서 `preprocess`가 반환한 딕셔너리가 에이전트의 입력이 돼요. 이는 LangChain의 LCEL(LangChain Expression Language) 파이프라인 합성 방식이에요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트가 이미 작성된 파일을 인식하고 이어서 작업할 수 있게 해줘요.
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → Supervisor → (DocWriter | NoteTaker | ChartGenerator | END) → Supervisor → ...
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. Super-Graph 구성: 계층 최상위 조립

이제 두 팀을 합쳐 Super-Graph를 만들 차례예요. 핵심은 **파이프라인 연산자(`|`)로 서브그래프를 합성**하는 패턴이에요.

```python
super_graph.add_node(
    "ResearchTeam",
    get_last_message | web_research_app | join_graph
)
```

이 코드는 세 단계를 파이프라인으로 연결해요:

| 단계 | 함수 | 역할 | 비유 |
|------|------|------|------|
| 1 | `get_last_message` | Super-Graph 상태에서 마지막 메시지만 추출 | 업무 지시서를 팀에게 전달 |
| 2 | `web_research_app` | Research Team 서브그래프 실행 | 팀 내부에서 작업 수행 |
| 3 | `join_graph` | 서브그래프 결과에서 마지막 메시지를 Super-Graph 상태로 병합 | 팀 결과 보고서를 CEO에게 올림 |

> 🔑 **핵심 개념**: `get_last_message`와 `join_graph`는 **상태 타입 변환 어댑터**예요. Super-Graph의 `SuperGraphState`와 서브그래프의 `ResearchTeamState`는 서로 다른 타입이에요. 이 두 함수가 타입 불일치를 자동으로 해결해주는 다리(bridge) 역할을 해요. 이 패턴을 기억하면 어떤 서브그래프도 Super-Graph에 쉽게 연결할 수 있어요.

> ⚠️ **자주 하는 실수**: 서브그래프를 Super-Graph에 직접 연결하면 상태 타입이 맞지 않아 에러가 발생해요. 반드시 `get_last_message | subgraph | join_graph` 3단계 파이프라인을 사용하세요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → Supervisor → (ResearchTeam | PaperWritingTeam | END) → Supervisor → ...
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 헬퍼 함수 및 실행

그래프를 실행하고 결과를 출력하는 헬퍼 함수를 정의할게요.

In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 6-1. Research Team 단독 실행 테스트

먼저 Research Team만 독립적으로 실행해서 동작을 확인해요.

> 💡 **핵심 정리**: 각 팀을 먼저 단독으로 테스트한 뒤 Super-Graph에 연결하는 것이 좋은 개발 전략이에요. 문제가 발생했을 때 어느 계층에서 발생했는지 바로 알 수 있어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 6-2. Super-Graph 전체 실행

이제 Research Team과 Doc Writing Team을 모두 활용하는 전체 워크플로를 실행해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Markdown

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import os

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. TODO 실습: 팀 추가하기

아래 실습에서는 새로운 **Review Team**을 Super-Graph에 추가해봐요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 새로운 팀을 Super-Graph에 추가해봐요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **계층적 에이전트 팀**: 단일 Supervisor 대신 Top Supervisor → Sub-Supervisor → Worker 3단계 계층으로 복잡한 작업을 분할하고 위임해요
- **AgentFactory 패턴**: 동일한 LLM 설정으로 여러 에이전트를 만들 때 팩토리 클래스를 사용하면 코드 중복을 줄이고 `name` 기반 추적이 가능해요
- **create_team_supervisor**: `RouteResponse(BaseModel)`과 `with_structured_output()`을 조합해 허용된 값만 반환하는 안정적인 라우팅 체인을 만들어요
- **preprocess | agent 패턴**: 파이프라인 연산자로 상태 전처리 함수를 에이전트 앞에 연결하면, 파일 목록 같은 컨텍스트를 에이전트에게 자동으로 주입할 수 있어요
- **get_last_message | subgraph | join_graph**: Super-Graph에서 서브그래프를 합성하는 표준 패턴으로, 상태 타입 불일치 문제를 깔끔하게 해결해요

## 다음 노트북 예고

다음 `01-Deep-Agents-Overview.ipynb` (Part 10)에서는 **Deep Agents** 프레임워크를 배워요. 지금까지 다진 멀티 에이전트 패턴(서브그래프, 핸드오프, Skills) 위에서 `create_deep_agent`로 장기 계획·서브에이전트·파일 시스템·메모리를 한 번에 갖춘 에이전트를 어떻게 만드는지 살펴볼게요.